In [ ]:
import os, time, json, requests, pandas as pd
from dataclasses import dataclass, asdict
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- site config --------------------
SITE_ROOT = "https://expinterweb.mites.gob.es"
BASE_URL = f"{SITE_ROOT}/regcon/pub/consultaPublicaEstatal"

def make_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    
    # Anti-bot mitigations for MITES
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    
    # Execute CDP command to hide webdriver flag from the site's JS
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver

# -------------------- small utils --------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def append_rows(out_dir, rows):
    """Append rows to CSV immediately to prevent data loss."""
    csv_path = os.path.join(out_dir, "spain_agreements.csv")
    df = pd.DataFrame([asdict(r) if hasattr(r, "__dict__") else r for r in rows])
    mode, header = ("a", False) if os.path.exists(csv_path) else ("w", True)
    df.to_csv(csv_path, mode=mode, header=header, index=False, encoding="utf-8")

# -------------------- parsing --------------------
@dataclass
class AgreementSpain:
    codigo: str | None
    denominacion: str | None
    tipo_tramite: str | None
    autoridad_laboral: str | None
    vigencia_desde: str | None
    vigencia_hasta: str | None
    detalle_url: str | None
    pdf_url: str | None
    pdf_saved_as: str | None

def download_pdf(url, out_dir, agreement_id):
    """Downloads the PDF and returns the local filename. Includes User-Agent to avoid blocks."""
    if not url: return None
    ensure_dir(out_dir)
    
    # Using the agreement ID as the filename to securely map metadata to the PDF
    name = f"{agreement_id}.pdf" 
    path = os.path.join(out_dir, name)
    
    if os.path.exists(path):   
        return path # skip re-download
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()
        if "application/pdf" in r.headers.get("Content-Type", "") or r.content.startswith(b"%PDF"):
            with open(path, "wb") as f:
                f.write(r.content)
            return path
        else:
            print(f"Warning: URL did not return a valid PDF: {url}")
            return None
    except Exception as e:
        print(f"Failed to download PDF {url}: {e}")
        return None

# -------------------- main scraper --------------------
def scrape_spain(out_dir="spain_data", target_count=10, pause=2.0):
    ensure_dir(out_dir)
    pdf_dir = os.path.join(out_dir, "pdfs")
    ensure_dir(pdf_dir)

    d = make_driver(headless=False) # Recommend False initially to bypass bot checks
    collected_agreements = []

    try:
        print(f"Loading {BASE_URL}...")
        d.get(BASE_URL)
        
        # 1. Trigger the search to load recent agreements
        # NOTE: You may need to inspect the live DOM to ensure this XPath matches the "Buscar" button.
        search_btn = WebDriverWait(d, 15).until(
            EC.element_to_be_clickable((By.XPATH, "//input[@value='Buscar'] | //button[contains(., 'Buscar')]"))
        )
        search_btn.click()
        time.sleep(pause)

        # 2. Wait for the results table to populate
        WebDriverWait(d, 15).until(EC.presence_of_element_located((By.XPATH, "//table//tr/td")))
        
        # 3. Parse the table rows
        rows = d.find_elements(By.XPATH, "//table//tr[td]")
        print(f"Found {len(rows)} agreements on the first page.")
        
        for row in rows[:target_count]:
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) < 7: continue
            
            # NOTE: These indices assume a standard REGCON column layout. 
            # Please adjust based on the actual table structure.
            codigo = cols[0].text.strip()
            
            detalle_link = None
            try:
                # Assuming the last column "Acciones" has a link to the detail page
                action_a = cols[-1].find_element(By.TAG_NAME, "a")
                detalle_link = action_a.get_attribute("href")
            except:
                pass

            ag = AgreementSpain(
                codigo=codigo,
                denominacion=cols[1].text.strip(),
                tipo_tramite=cols[2].text.strip(),
                autoridad_laboral=cols[3].text.strip(),
                vigencia_desde=cols[4].text.strip(),
                vigencia_hasta=cols[5].text.strip(),
                detalle_url=detalle_link,
                pdf_url=None,
                pdf_saved_as=None
            )
            collected_agreements.append(ag)

        # 4. Visit each detail page to extract the actual PDF link
        for ag in collected_agreements:
            if not ag.detalle_url:
                continue
                
            print(f"Fetching details for {ag.codigo}...")
            d.get(ag.detalle_url)
            time.sleep(pause) # Crucial for not getting IP banned by MITES
            
            try:
                # Look for a link ending in .pdf or containing the word 'Descargar'/'PDF'
                pdf_a = d.find_element(By.XPATH, "//a[contains(@href, '.pdf') or contains(translate(text(), 'PDF', 'pdf'), 'pdf')]")
                ag.pdf_url = pdf_a.get_attribute("href")
                
                # Download and map PDF
                if ag.pdf_url:
                    ag.pdf_saved_as = download_pdf(ag.pdf_url, pdf_dir, ag.codigo)
            except Exception as e:
                print(f"Could not find PDF link for {ag.codigo}")
            
            # Save progress immediately
            append_rows(out_dir, [ag])

        print(f"Successfully processed {len(collected_agreements)} agreements for prototype.")
        
    finally:
        d.quit()

    # Return summary dataframe
    csv_path = os.path.join(out_dir, "spain_agreements.csv")
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()

# To run the prototype:
# df = scrape_spain(target_count=10)
# print(df.head())